# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab

In [ ]:
%%shell

RUTA_DMEYF="/content/.drive/My Drive/Capacitacion/Maestria Data Minning/09-Aplicaciones_mineria_datos_en_economia/dmeyf"

mkdir -p "$RUTA_DMEYF"
mkdir -p "/content/buckets"
ln -sfn "$RUTA_DMEYF" /content/buckets/b1

mkdir -p ~/.kaggle
cp "$RUTA_DMEYF/kaggle/kaggle.json" ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget "$url_origen""$archivo" -O "$carpeta_destino""$archivo"
  fi

  if ! test -f "/content/datasets/""$archivo"; then
    cp "$carpeta_destino""$archivo" "/content/datasets/""$archivo"
  fi
}

descargar "dataset_pequeno.csv"

---

## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta corrida se construira cada arbol utilizando la fraccion de campos que resulte del grid search

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

Reconstruyo el ambiente (Drive, symlink, kaggle.json, dataset local) porque el cambio de runtime arranca una VM nueva

In [ ]:
RUTA_DMEYF <- "/content/.drive/My Drive/Capacitacion/Maestria Data Minning/09-Aplicaciones_mineria_datos_en_economia/dmeyf"

system(paste0(
  "python3 -c \"from google.colab import drive; drive.mount('/content/.drive')\""
), intern = TRUE)

system("mkdir -p /content/buckets", intern = TRUE)
system(paste0('ln -sfn "', RUTA_DMEYF, '" /content/buckets/b1'), intern = TRUE)

system("mkdir -p ~/.kaggle", intern = TRUE)
system("cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle", intern = TRUE)
system("chmod 600 ~/.kaggle/kaggle.json", intern = TRUE)

system("mkdir -p /content/datasets", intern = TRUE)
system("cp /content/buckets/b1/datasets/dataset_pequeno.csv /content/datasets/dataset_pequeno.csv", intern = TRUE)

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("pROC")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 100043

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4210_GridSearch_ArbolesAzarosos"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

### Grid Search de los 5 hiperparametros

Arranco probando los valores extremos de cada hiperparametro (feature_fraction, cp, maxdepth, minsplit, minbucket), dejando los otros cuatro fijos en un valor intermedio. Me quedo con el extremo que mejor AUC da en el holdout, y en la siguiente ronda achico el rango alrededor de ese valor. Repito unas vueltas hasta que el rango converge cerca del optimo.

In [ ]:
n_total <- nrow(dtrain)
n_fit <- as.integer(n_total * 0.8)
idx <- sample(1:n_total)
dtrain_fit <- dtrain[idx[1:n_fit]]
dvalid <- dtrain[idx[(n_fit + 1):n_total]]
dvalid[, clase_binaria := ifelse(clase_ternaria == "BAJA+2", 1, 0)]

evaluar <- function(combo) {
  prob <- rep(0, nrow(dvalid))
  for (arbolito in seq(32)) {
    qty_campos <- as.integer(length(campos_buenos) * combo$feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos)
    campos_random <- paste(campos_random, collapse = " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita,
      data = dtrain_fit,
      xval = 0,
      control = list(
        cp = combo$cp,
        minsplit = combo$minsplit,
        minbucket = min(combo$minbucket, combo$minsplit),
        maxdepth = combo$maxdepth
      )
    )

    pred <- predict(modelo, dvalid, type = "prob")
    prob <- prob + pred[, "BAJA+2"]
  }
  as.numeric(pROC::auc(dvalid$clase_binaria, prob, quiet = TRUE))
}

In [ ]:
rango <- list(
  feature_fraction = c(0.2, 0.8),
  cp = c(-1, 0.02),
  maxdepth = c(6, 30),
  minsplit = c(10, 600),
  minbucket = c(2, 200)
)

base <- list(feature_fraction = 0.5, cp = -1, maxdepth = 20, minsplit = 100, minbucket = 30)

archivo_log <- "grid_hiperparametros_log.csv"
log_grid <- if (file.exists(archivo_log)) fread(archivo_log) else
  data.table(ronda = integer(), parametro = character(), valor = numeric(), auc = numeric())

set.seed(PARAM$semilla_primigenia)

for (ronda_actual in 1:4) {
  for (nombre in names(rango)) {
    aucs <- numeric(0)
    for (v in rango[[nombre]]) {
      ya <- log_grid[ronda == ronda_actual & parametro == nombre & round(valor, 6) == round(v, 6)]
      if (nrow(ya) > 0) {
        auc <- ya$auc[1]
      } else {
        combo <- base
        combo[[nombre]] <- v
        auc <- evaluar(combo)
        log_grid <- rbind(log_grid, data.table(ronda = ronda_actual, parametro = nombre, valor = v, auc = auc))
        fwrite(log_grid, archivo_log)
      }
      aucs <- c(aucs, auc)
      cat(sprintf("ronda %d | %s = %.4f -> AUC %.4f\n", ronda_actual, nombre, v, auc))
    }
    base[[nombre]] <- rango[[nombre]][which.max(aucs)]
  }

  for (nombre in names(rango)) {
    centro <- base[[nombre]]
    ancho <- diff(range(rango[[nombre]]))
    rango[[nombre]] <- c(centro - ancho / 4, centro + ancho / 4)
  }
  rango$maxdepth <- pmin(pmax(round(rango$maxdepth), 2), 30)
  rango$minsplit <- pmax(round(rango$minsplit), 2)
  rango$minbucket <- pmax(round(rango$minbucket), 1)
  rango$feature_fraction <- pmin(pmax(rango$feature_fraction, 0.1), 0.9)
}

In [ ]:
PARAM$feature_fraction <- base$feature_fraction

PARAM$rpart$cp <- base$cp
PARAM$rpart$minsplit <- round(base$minsplit)
PARAM$rpart$minbucket <- min(round(base$minbucket), round(base$minsplit))
PARAM$rpart$maxdepth <- min(round(base$maxdepth), 30)

PARAM$num_trees_max <- 32

In [ ]:
# que tamanos de ensemble grabo a disco
grabar <- c(32)

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
tb_prediccion[, prob_acumulada := 0]

In [ ]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [ ]:
for (arbolito in seq(PARAM$num_trees_max) ) {
  message( arbolito, " ")
  qty_campos_a_utilizar <- as.integer(length(campos_buenos)
    * PARAM$feature_fraction)

  # elijo los campos al azar
  campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

  # paso de un vector a un string con los elementos
  # separados por un signo de "+"
  # este hace falta para la formula
  campos_random <- paste(campos_random, collapse= " + ")

  # armo la formula para rpart
  formulita <- paste0("clase_ternaria ~ ", campos_random)

  # genero el arbol de decision
  modelo <- rpart(formulita,
    data= dtrain,
    xval= 0,
    control= PARAM$rpart
  )

  # aplico el modelo a los datos que no tienen clase
  prediccion <- predict(modelo, dfuture, type= "prob")

  tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

  if (arbolito %in% grabar) {
    umbral_corte <- (1 / 40) * arbolito
    tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

    archivo_kaggle <- paste0(
        "KA421_",
        sprintf("%.3d", arbolito), # para que tenga ceros adelante
        ".csv"
      )

    # grabo el archivo
    fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle,
      sep= ","
    )

    # subida a Kaggle
    comando <- "kaggle competitions submit"
    competencia <- "-c utn-2026-inicial"
    arch <- paste( "-f", archivo_kaggle)

    mensaje <- paste0("-m 'cp=", PARAM$rpart$cp, "  minsplit=", PARAM$rpart$minsplit, "  minbucket=", PARAM$rpart$minbucket, " maxdepth=", PARAM$rpart$maxdepth, "'" )
    linea <- paste( comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    cat(salida)
  }
}

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
str(PARAM$rpart)

---